In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, MultiProductContextEmbeddings
from src.utils import TemporalSplitter

/home/thebigmonster/Github/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Datos ──────────────────────────────────────────────────────────
N_UPCS = 5
SMOOTH_WINDOW = 8
BETA_EDA = -2

# ── Tuning robusto ─────────────────────────────────────────────────
N_FOLDS = 3
TUNE_SEEDS = [11, 29, 42]
MIN_TRAIN_FRAC = 0.50

# ── Entrenamiento para tuning ──────────────────────────────────────
N_EPOCHS_P0 = 200
N_EPOCHS_P1 = 200
N_EPOCHS_P2 = 250
PATIENCE    = 20
ES_PATIENCE = 40

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Resultados ─────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_TRIAL_PATH = RESULTS_DIR / "best_trial_params.json"
TRIAL_SUMMARY_PATH = RESULTS_DIR / "nn_hparam_trials_summary.csv"

Device: cuda


In [4]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

In [5]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
print(f"Tiendas: {n_stores}  |  Semanas: {n_weeks}")

mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)

full_wide_raw = mp_builder.transform(df).copy()
n_upcs = mp_builder.n

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs seleccionados: {n_upcs}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 30)
Tiendas: 70  |  Semanas: 302
Full wide shape: (19808, 101)
UPCs seleccionados: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


In [6]:
splitter = TemporalSplitter(week_col="week_id")
fold_splits = splitter.expanding_splits(
    df=full_wide_raw,
    n_folds=N_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

print(f"N folds disponibles: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i}: train={len(train_fold):,} "
        f"val={len(val_fold):,} "
        f"train_weeks={train_fold['week_id'].nunique()} "
        f"val_weeks={val_fold['week_id'].nunique()}"
    )

N folds disponibles: 3
Fold 0: train=9,756 val=3,394 train_weeks=151 val_weeks=50
Fold 1: train=13,150 val=3,339 train_weeks=201 val_weeks=50
Fold 2: train=16,489 val=3,254 train_weeks=251 val_weeks=50


In [7]:
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}

def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s


def build_fold_datasets(train_wide, val_wide, train_wide_s, val_wide_s):
    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)
    return train_ds_p0, val_ds_p0, train_ds, val_ds

In [8]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, phase_name="", verbose=False):

    best_val_loss = float("inf")
    no_improve    = 0
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, eps_hat, aux = model(batch, return_parts=True)
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                        aux["Bx"], aux["IBx"],
                                        model.head.param_head._pairs)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, eps_hat, aux = model(batch, return_parts=True)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                    aux["Bx"], aux["IBx"],
                                    model.head.param_head._pairs)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_denom = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device) for k, v in batch.items()}
                y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
                obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

                y_hat, eps_hat, aux = model(batch, return_parts=True)
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                  aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                  aux["Bx"], aux["IBx"],
                                  model.head.param_head._pairs)
                denom        = obs_mask.sum().item()
                val_loss_sum += logs["loss"].item() * denom
                val_denom    += denom

        val_loss = val_loss_sum / max(val_denom, 1.0)
        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping en época {epoch+1}")
            break

    return best_val_loss

print("run_training definida")

run_training definida


In [9]:
HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64_32":  (128, 64, 32),
    "64_32_16":   (64, 32, 16),
    "128_64":     (128, 64),
}

def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)
            y_hat, _, _ = model(batch, return_parts=True)

            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())

    y_true_all = torch.cat(all_true).float()
    y_pred_all = torch.cat(all_pred).float()

    err = y_true_all - y_pred_all
    mae = float(err.abs().mean())
    rmse = float(torch.sqrt((err ** 2).mean()))

    ss_res = float((err ** 2).sum())
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum())
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }


def compute_elasticity_score(model, val_loader, device, elast_min=-5.0, elast_max=0.0):
    model.eval()
    all_elast = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            obs_mask = torch.stack([batch[f"obs_mask_{i}"] for i in range(model.n)], dim=1).bool()
            _, eps_hat, _ = model(batch, return_parts=True)
            all_elast.append(eps_hat[obs_mask].cpu())

    elast = torch.cat(all_elast).numpy()

    in_range = float(((elast >= elast_min) & (elast <= elast_max)).mean())
    median_e = float(np.median(elast))

    deviation = max(0.0, abs(median_e - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)

    score = in_range * (1.0 - prior_penalty)

    return {
        "elast_score": float(score),
        "elasticity_median": median_e,
        "elasticity_in_range": float(in_range),
    }

print("Helpers de métricas definidos")

Helpers de métricas definidos


In [10]:
def build_and_train(params, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed)

    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold,
        val_wide=val_fold,
        smooth_window=SMOOTH_WINDOW,
    )

    train_ds_p0, val_ds_p0, train_ds, val_ds = build_fold_datasets(
        train_wide, val_wide, train_wide_s, val_wide_s
    )

    n_knots          = params["N_KNOTS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout          = params["DROPOUT"]
    d_store          = params.get("D_STORE", 16)
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lr_p2            = params["LR_P2"]
    lambda_smooth_p2 = params["LAMBDA_SMOOTH_P2"]
    lambda_pos_p2    = params["LAMBDA_POS_P2"]
    batch_size       = params["BATCH_SIZE"]

    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase1.pt"
    ckpt_p2 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase2.pt"

    loader_factory = DataLoaderFactory(num_workers=0, pin_memory=True)
    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader_p0 = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=batch_size, shuffle=False
    )
    train_loader = loader_factory.create_train_loader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader = loader_factory.create_eval_loader(
        val_ds, batch_size=batch_size, shuffle=False
    )

    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    cb = MultiProductContextEmbeddings(
        n=n_upcs,
        n_stores=n_stores,
        d_store=d_store,
    )

    def make_model(enforce_negative_beta, use_cross):
        head = IntegrableDemandHead(
            context_dim=cb.out_dim,
            K_splines=n_knots,
            n=n_upcs,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        return ICDN(
            context_builder=cb,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── FASE 0 ─────────────────────────────────────────────────────
    m0 = make_model(enforce_negative_beta=True, use_cross=False)
    with torch.no_grad():
        m0.head.param_head.head_w.weight.zero_()
        m0.head.param_head.head_w.bias.zero_()
    m0.head.param_head.head_w.weight.requires_grad_(False)
    m0.head.param_head.head_w.bias.requires_grad_(False)

    beta_raw_init = torch.log(torch.exp(torch.tensor(-BETA_EDA, dtype=torch.float32)) - 1.0)
    with torch.no_grad():
        m0.head.param_head.head_beta.weight.zero_()
        m0.head.param_head.head_beta.bias.fill_(beta_raw_init)

    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE, ckpt_p0, device, "P0")

    # ── FASE 1 ─────────────────────────────────────────────────────
    m1 = make_model(enforce_negative_beta=True, use_cross=False)
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    m1.head.param_head.head_w.weight.requires_grad_(True)
    m1.head.param_head.head_w.bias.requires_grad_(True)

    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE, ckpt_p1, device, "P1")

    # ── FASE 2 ─────────────────────────────────────────────────────
    m2 = make_model(enforce_negative_beta=True, use_cross=True)
    state = torch.load(ckpt_p1, map_location=device)
    state.pop("head.param_head._pairs", None)
    m2.load_state_dict(state, strict=False)

    m2.head.param_head.head_w.weight.requires_grad_(True)
    m2.head.param_head.head_w.bias.requires_grad_(True)
    with torch.no_grad():
        m2.head.param_head.head_cross.weight.zero_()
        m2.head.param_head.head_cross.bias.zero_()

    loss_p2 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth_p2,
        lambda_pos=lambda_pos_p2,
        reduction="none",
    )
    decay, no_decay = [], []
    for name, p in m2.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p2 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p2,
    )
    sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p2, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m2, train_loader, val_loader, loss_p2,
                 opt_p2, sch_p2, N_EPOCHS_P2, ES_PATIENCE, ckpt_p2, device, "P2")

    m2.load_state_dict(torch.load(ckpt_p2, map_location=device))

    pred_metrics = compute_global_metrics(m2, val_loader, device)
    elast_metrics = compute_elasticity_score(m2, val_loader, device)

    out = {
        "trial_id": trial_id,
        "fold": fold_id,
        "seed": seed,
        "n_train": len(train_wide),
        "n_val": len(val_wide),
        **pred_metrics,
        **elast_metrics,
    }

    print(
        f"trial={trial_id} fold={fold_id} seed={seed} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} "
        f"ElastScore={out['elast_score']:.4f}"
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    ckpt_p2.unlink(missing_ok=True)

    return out

print("build_and_train redefinida")

build_and_train redefinida


In [11]:
trial_records = []

def objective(trial):
    params = {
        "N_KNOTS":          trial.suggest_int("N_KNOTS", 2, 16),
        "HIDDEN_KEY":       trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":          trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":            trial.suggest_float("LR_P0",  1e-4, 1e-2, log=True),
        "LR_P1":            trial.suggest_float("LR_P1",  1e-5, 5e-3, log=True),
        "LR_P2":            trial.suggest_float("LR_P2",  1e-5, 1e-3, log=True),
        "LAMBDA_SMOOTH_P2": trial.suggest_float("LAMBDA_SMOOTH_P2", 1e-5, 0.2, log=True),
        "LAMBDA_POS_P2":    trial.suggest_float("LAMBDA_POS_P2",    0.05, 0.5),
        "BATCH_SIZE":       trial.suggest_categorical("BATCH_SIZE", [16, 32, 64]),
    }

    print(f"\n{'='*70}")
    print(f"Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*70}")

    run_rows = []
    for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            row = build_and_train(
                params=params,
                train_fold=train_fold,
                val_fold=val_fold,
                fold_id=fold_id,
                seed=seed,
                trial_id=trial.number,
            )
            run_rows.append(row)

    df_trial = pd.DataFrame(run_rows)

    mean_r2 = float(df_trial["r2_val"].mean())
    std_r2  = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    mean_elast = float(df_trial["elast_score"].mean())
    std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    mean_mae  = float(df_trial["mae_val"].mean())
    mean_rmse = float(df_trial["rmse_val"].mean())

    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("mean_mae", mean_mae)
    trial.set_user_attr("mean_rmse", mean_rmse)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("robust_elast", robust_elast)

    df_trial["trial"] = trial.number
    for k, v in params.items():
        df_trial[k] = v
    trial_records.extend(df_trial.to_dict(orient="records"))

    print(
        f"Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast={mean_elast:.4f} std_Elast={std_elast:.4f} "
        f"robust_Elast={robust_elast:.4f}"
    )

    return robust_r2, robust_elast

In [12]:
study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="hparam_pareto_kfold_seed",
    storage="sqlite:///../results/hparam_pareto_kfold_seed.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=10)

print(f"\nTrials completados: {len(study.trials)}")
print(f"Trials Pareto-óptimos: {len(study.best_trials)}")

[I 2026-03-21 19:20:39,768] Using an existing study with name 'hparam_pareto_kfold_seed' instead of creating a new one.



Trial 34
  N_KNOTS: 12
  HIDDEN_KEY: 64_32
  DROPOUT: 0.02719900146420525
  LR_P0: 0.00020303343841063246
  LR_P1: 0.00259765191490959
  LR_P2: 0.000406152719980858
  LAMBDA_SMOOTH_P2: 0.1649755752037984
  LAMBDA_POS_P2: 0.210758666555927
  BATCH_SIZE: 16
trial=34 fold=0 seed=11 | R2=0.7326 MAE=0.5136 ElastScore=0.2081
trial=34 fold=0 seed=29 | R2=0.7046 MAE=0.5308 ElastScore=0.1628
trial=34 fold=0 seed=42 | R2=0.7044 MAE=0.5326 ElastScore=0.1464
trial=34 fold=1 seed=11 | R2=0.3852 MAE=0.6726 ElastScore=0.8488
trial=34 fold=1 seed=29 | R2=0.6355 MAE=0.5147 ElastScore=0.9611
trial=34 fold=1 seed=42 | R2=0.6567 MAE=0.5099 ElastScore=0.9361
trial=34 fold=2 seed=11 | R2=0.4236 MAE=0.5122 ElastScore=0.8090
trial=34 fold=2 seed=29 | R2=0.3699 MAE=0.5343 ElastScore=0.7741


[I 2026-03-22 03:02:16,130] Trial 34 finished with values: [0.5130509339409809, 0.5357017062601513] and parameters: {'N_KNOTS': 12, 'HIDDEN_KEY': '64_32', 'DROPOUT': 0.02719900146420525, 'LR_P0': 0.00020303343841063246, 'LR_P1': 0.00259765191490959, 'LR_P2': 0.000406152719980858, 'LAMBDA_SMOOTH_P2': 0.1649755752037984, 'LAMBDA_POS_P2': 0.210758666555927, 'BATCH_SIZE': 16}.


trial=34 fold=2 seed=42 | R2=0.3680 MAE=0.5437 ElastScore=0.7494
Trial 34 summary | mean_R2=0.5534 std_R2=0.1614 robust_R2=0.5131 | mean_Elast=0.6217 std_Elast=0.3442 robust_Elast=0.5357

Trial 35
  N_KNOTS: 6
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.13144995317161684
  LR_P0: 0.00175991320956907
  LR_P1: 0.00031637599672751935
  LR_P2: 3.4762405644567195e-05
  LAMBDA_SMOOTH_P2: 0.04815859111272099
  LAMBDA_POS_P2: 0.2979090163627111
  BATCH_SIZE: 32
trial=35 fold=0 seed=11 | R2=0.7145 MAE=0.5259 ElastScore=0.9124
trial=35 fold=0 seed=29 | R2=0.7272 MAE=0.5123 ElastScore=0.9899
trial=35 fold=0 seed=42 | R2=0.7263 MAE=0.5134 ElastScore=0.7653
trial=35 fold=1 seed=11 | R2=-2.5176 MAE=0.6883 ElastScore=0.5830
trial=35 fold=1 seed=29 | R2=0.4254 MAE=0.5802 ElastScore=0.3346
trial=35 fold=1 seed=42 | R2=0.6082 MAE=0.5384 ElastScore=0.5373
trial=35 fold=2 seed=11 | R2=0.4580 MAE=0.5041 ElastScore=1.0000
trial=35 fold=2 seed=29 | R2=0.4572 MAE=0.5029 ElastScore=1.0000


[I 2026-03-22 07:44:45,790] Trial 35 finished with values: [-0.02967476453907214, 0.728776668671991] and parameters: {'N_KNOTS': 6, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.13144995317161684, 'LR_P0': 0.00175991320956907, 'LR_P1': 0.00031637599672751935, 'LR_P2': 3.4762405644567195e-05, 'LAMBDA_SMOOTH_P2': 0.04815859111272099, 'LAMBDA_POS_P2': 0.2979090163627111, 'BATCH_SIZE': 32}.


...

trial=41 fold=2 seed=42 | R2=0.5151 MAE=0.4818 ElastScore=0.3730
Trial 41 summary | mean_R2=0.6428 std_R2=0.1022 robust_R2=0.6173 | mean_Elast=0.6164 std_Elast=0.1641 robust_Elast=0.5753

Trial 42
  N_KNOTS: 16
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.06248594227234833
  LR_P0: 0.0001289675283646261
  LR_P1: 2.4834872853845825e-05
  LR_P2: 5.1953922795920056e-05
  LAMBDA_SMOOTH_P2: 7.297598537724831e-05
  LAMBDA_POS_P2: 0.3946706323688513
  BATCH_SIZE: 32
trial=42 fold=0 seed=11 | R2=0.7597 MAE=0.4823 ElastScore=0.4231
trial=42 fold=0 seed=29 | R2=0.7448 MAE=0.4997 ElastScore=0.5593
trial=42 fold=0 seed=42 | R2=0.7616 MAE=0.4812 ElastScore=0.4777
trial=42 fold=1 seed=11 | R2=0.6433 MAE=0.5176 ElastScore=0.5791
trial=42 fold=1 seed=29 | R2=0.6444 MAE=0.5166 ElastScore=0.5814
trial=42 fold=1 seed=42 | R2=0.6296 MAE=0.5197 ElastScore=0.5182
trial=42 fold=2 seed=11 | R2=0.5065 MAE=0.4853 ElastScore=0.7620
trial=42 fold=2 seed=29 | R2=0.5196 MAE=0.4777 ElastScore=0.8108


[I 2026-03-23 15:01:09,908] Trial 42 finished with values: [0.6096177209521793, 0.5604756067719467] and parameters: {'N_KNOTS': 16, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.06248594227234833, 'LR_P0': 0.0001289675283646261, 'LR_P1': 2.4834872853845825e-05, 'LR_P2': 5.1953922795920056e-05, 'LAMBDA_SMOOTH_P2': 7.297598537724831e-05, 'LAMBDA_POS_P2': 0.3946706323688513, 'BATCH_SIZE': 32}.


trial=42 fold=2 seed=42 | R2=0.5136 MAE=0.4810 ElastScore=0.6146
Trial 42 summary | mean_R2=0.6359 std_R2=0.1051 robust_R2=0.6096 | mean_Elast=0.5918 std_Elast=0.1254 robust_Elast=0.5605

Trial 43
  N_KNOTS: 11
  HIDDEN_KEY: 128_64
  DROPOUT: 0.2567160995398289
  LR_P0: 0.0018029081160086519
  LR_P1: 1.4779924383440461e-05
  LR_P2: 1.565099692883302e-05
  LAMBDA_SMOOTH_P2: 0.04891370302224874
  LAMBDA_POS_P2: 0.15479429181913873
  BATCH_SIZE: 64
trial=43 fold=0 seed=11 | R2=0.7072 MAE=0.5346 ElastScore=0.3842
trial=43 fold=0 seed=29 | R2=0.7141 MAE=0.5269 ElastScore=0.4617
trial=43 fold=0 seed=42 | R2=0.7064 MAE=0.5336 ElastScore=0.4121
trial=43 fold=1 seed=11 | R2=0.5815 MAE=0.5559 ElastScore=0.3020
trial=43 fold=1 seed=29 | R2=0.6079 MAE=0.5398 ElastScore=0.2900
trial=43 fold=1 seed=42 | R2=0.5252 MAE=0.5716 ElastScore=0.5873
trial=43 fold=2 seed=11 | R2=0.4340 MAE=0.5124 ElastScore=0.8308
trial=43 fold=2 seed=29 | R2=0.4480 MAE=0.5097 ElastScore=0.7780


[I 2026-03-23 18:41:56,383] Trial 43 finished with values: [0.5418313683722153, 0.49117671375296773] and parameters: {'N_KNOTS': 11, 'HIDDEN_KEY': '128_64', 'DROPOUT': 0.2567160995398289, 'LR_P0': 0.0018029081160086519, 'LR_P1': 1.4779924383440461e-05, 'LR_P2': 1.565099692883302e-05, 'LAMBDA_SMOOTH_P2': 0.04891370302224874, 'LAMBDA_POS_P2': 0.15479429181913873, 'BATCH_SIZE': 64}.


trial=43 fold=2 seed=42 | R2=0.4237 MAE=0.5168 ElastScore=0.9017
Trial 43 summary | mean_R2=0.5720 std_R2=0.1207 robust_R2=0.5418 | mean_Elast=0.5498 std_Elast=0.2343 robust_Elast=0.4912

Trials completados: 44
Trials Pareto-óptimos: 4


In [13]:
summary_rows = []
for t in study.trials:
    if t.values is None:
        continue
    row = {
        "trial": t.number,
        "mean_r2": t.user_attrs.get("mean_r2", np.nan),
        "std_r2": t.user_attrs.get("std_r2", np.nan),
        "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
        "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
        "mean_mae": t.user_attrs.get("mean_mae", np.nan),
        "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
        **t.params,
    }
    summary_rows.append(row)

df_trials_summary = pd.DataFrame(summary_rows).sort_values(
    ["mean_r2", "mean_elast_score"], ascending=[False, False]
)

print(df_trials_summary.head(15).to_string(index=False))

 trial  mean_r2   std_r2  mean_elast_score  std_elast_score  mean_mae  mean_rmse  N_KNOTS HIDDEN_KEY  DROPOUT    LR_P0    LR_P1    LR_P2  LAMBDA_SMOOTH_P2  LAMBDA_POS_P2  BATCH_SIZE
    41 0.642813 0.102166          0.616373         0.164099  0.491395   0.633481        3      64_32 0.166660 0.004896 0.000045 0.000665          0.000013       0.263288          64
    12 0.642596 0.094238          0.650589         0.145541  0.491504   0.635390       14     128_64 0.162260 0.001795 0.000049 0.000450          0.000030       0.458544          64
    26 0.637644 0.101385          0.524480         0.156640  0.492553   0.638141        6      64_32 0.130027 0.000364 0.000023 0.000479          0.000011       0.159380          32
    38 0.636902 0.103624          0.629717         0.090319  0.496113   0.638511       14   64_32_16 0.121325 0.002821 0.000059 0.000121          0.000023       0.317597          64
    42 0.635897 0.105118          0.591813         0.125350  0.495685   0.638608       16 

In [14]:
df_trials_summary["robust_score"] = (
    df_trials_summary["mean_r2"]
    - 0.25 * df_trials_summary["std_r2"].fillna(0.0)
    + 0.10 * df_trials_summary["mean_elast_score"]
)

best_row = df_trials_summary.sort_values("robust_score", ascending=False).iloc[0]

best_trial_payload = {
    "trial": int(best_row["trial"]),
    "robust_score": float(best_row["robust_score"]),
    "mean_r2": float(best_row["mean_r2"]),
    "std_r2": float(best_row["std_r2"]),
    "mean_elast_score": float(best_row["mean_elast_score"]),
    "std_elast_score": float(best_row["std_elast_score"]),
    "params": {
        "N_KNOTS": int(best_row["N_KNOTS"]),
        "HIDDEN_KEY": str(best_row["HIDDEN_KEY"]),
        "DROPOUT": float(best_row["DROPOUT"]),
        "LR_P0": float(best_row["LR_P0"]),
        "LR_P1": float(best_row["LR_P1"]),
        "LR_P2": float(best_row["LR_P2"]),
        "LAMBDA_SMOOTH_P2": float(best_row["LAMBDA_SMOOTH_P2"]),
        "LAMBDA_POS_P2": float(best_row["LAMBDA_POS_P2"]),
        "BATCH_SIZE": int(best_row["BATCH_SIZE"]),
    }
}

with open(BEST_TRIAL_PATH, "w", encoding="utf-8") as f:
    json.dump(best_trial_payload, f, indent=2, ensure_ascii=False)

df_trials_summary.to_csv(TRIAL_SUMMARY_PATH, index=False)

print("Best trial guardado en:", BEST_TRIAL_PATH)
print("Resumen trials guardado en:", TRIAL_SUMMARY_PATH)
print(json.dumps(best_trial_payload, indent=2, ensure_ascii=False))

Best trial guardado en: ../results/best_trial_params.json
Resumen trials guardado en: ../results/nn_hparam_trials_summary.csv
{
  "trial": 14,
  "robust_score": 0.6898868011601498,
  "mean_r2": 0.625558203160165,
  "std_r2": 0.11445097301152933,
  "mean_elast_score": 0.9294134125286713,
  "std_elast_score": 0.11389084110445409,
  "params": {
    "N_KNOTS": 10,
    "HIDDEN_KEY": "64_32_16",
    "DROPOUT": 0.17644785552418754,
    "LR_P0": 0.0006963825561558388,
    "LR_P1": 0.00011113650292557837,
    "LR_P2": 0.00020465153569806238,
    "LAMBDA_SMOOTH_P2": 0.01603732582410331,
    "LAMBDA_POS_P2": 0.2781456207286023,
    "BATCH_SIZE": 16
  }
}
